In [1]:
import warnings, os, json, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix,
    roc_curve, precision_recall_curve, average_precision_score,
)

import xgboost as xgb
import lightgbm as lgb
import optuna
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.base import clone
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [2]:
TRAIN_PATH  = "Leads_Selected_Train.csv"
TEST_PATH   = "Leads_Selected_Test.csv"
TARGET_COL  = "Converted"
OUTPUT_DIR  = "outputs"
N_TRIALS    = 200    
CV_FOLDS    = 5
SEED        = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(SEED)

cat_cols = ['Lead Origin', 'Last Activity', 'What is your current occupation', 'Lead Profile'] 

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ],
    remainder='passthrough'
)

In [3]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

FEATURE_NAMES = train_df.drop(columns=[TARGET_COL]).columns.tolist()

X_train = train_df[FEATURE_NAMES].values
y_train = train_df[TARGET_COL].values
X_test  = test_df[FEATURE_NAMES].values
y_test  = test_df[TARGET_COL].values

neg, pos    = (y_train == 0).sum(), (y_train == 1).sum()
SPW         = neg / pos   

print(f"Train : {X_train.shape}  |  Test : {X_test.shape}")
print(f"Class 0 : {neg}  |  Class 1 : {pos}  |  SPW : {SPW:.3f}")
print()

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

Train : (6152, 6)  |  Test : (1538, 6)
Class 0 : 3699  |  Class 1 : 2453  |  SPW : 1.508



THRESHOLD: dùng Out-of-Fold predictions, Tìm threshold trên OOF thay vì full train, tránh overfit threshold

In [4]:
def find_best_threshold_oof(model, X, y, build_fn, params, fit_kwargs=None):
    """
    Tạo OOF predictions bằng StratifiedKFold,
    sau đó quét threshold [0.10, 0.90] để maximize Accuracy.
    """
    fit_kwargs = fit_kwargs or {}
    oof_prob   = np.zeros(len(y))

    for tr_idx, val_idx in cv.split(X, y):
        Xtr, Xval = X[tr_idx], X[val_idx]
        ytr, yval = y[tr_idx], y[val_idx]
        m = build_fn(params)
        m.fit(Xtr, ytr, **fit_kwargs)
        oof_prob[val_idx] = m.predict_proba(Xval)[:, 1]

    thresholds = np.linspace(0.10, 0.90, 161)
    accs       = [accuracy_score(y, (oof_prob >= t).astype(int))
                  for t in thresholds]
    best_idx   = int(np.argmax(accs))
    return thresholds[best_idx], accs[best_idx], oof_prob

OPTUNA OBJECTIVES: tune ROC-AUC trên Stratified CV

In [5]:
def objective_xgb(trial):
    params = dict(
        n_estimators      = trial.suggest_int("n_estimators", 200, 1000),
        max_depth         = trial.suggest_int("max_depth", 3, 10),
        learning_rate     = trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        subsample         = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree  = trial.suggest_float("colsample_bytree", 0.4, 1.0),
        colsample_bylevel = trial.suggest_float("colsample_bylevel", 0.4, 1.0),
        min_child_weight  = trial.suggest_int("min_child_weight", 1, 20),
        gamma             = trial.suggest_float("gamma", 0.0, 5.0),
        reg_alpha         = trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda        = trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        scale_pos_weight  = trial.suggest_float("scale_pos_weight", SPW * 0.5, SPW * 2.0),
        eval_metric       = "auc",
        early_stopping_rounds = 40,
        tree_method       = "hist",
        enable_categorical= True,
        random_state      = SEED, 
        n_jobs            = -1, 
        verbosity         = 0
    )
    
    aucs = []
    
    for tr_idx, val_idx in cv.split(X_train, y_train):
        if isinstance(X_train, pd.DataFrame):
            Xtr, Xval = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        else:
            Xtr, Xval = X_train[tr_idx], X_train[val_idx]
            
        ytr, yval = y_train[tr_idx], y_train[val_idx]
        
        m = xgb.XGBClassifier(**params)
        m.fit(Xtr, ytr, eval_set=[(Xval, yval)], verbose=False)
        aucs.append(roc_auc_score(yval, m.predict_proba(Xval)[:, 1]))
        
    return np.mean(aucs)

In [6]:
def objective_lgb(trial):
    params = dict(
        n_estimators      = trial.suggest_int("n_estimators", 200, 1000),
        max_depth         = trial.suggest_int("max_depth", 3, 12),
        num_leaves        = trial.suggest_int("num_leaves", 20, 200), 
        
        learning_rate     = trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        min_child_samples = trial.suggest_int("min_child_samples", 5, 100),
        subsample         = trial.suggest_float("subsample", 0.5, 1.0),
        subsample_freq    = trial.suggest_int("subsample_freq", 1, 10),
        colsample_bytree  = trial.suggest_float("colsample_bytree", 0.4, 1.0),
        reg_alpha         = trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda        = trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        min_split_gain    = trial.suggest_float("min_split_gain", 0.0, 1.0),
        is_unbalance      = True,
        random_state      = SEED, 
        n_jobs            = -1, 
        verbosity         = -1,
    )
    
    aucs = []
    for tr_idx, val_idx in cv.split(X_train, y_train):
        if isinstance(X_train, pd.DataFrame):
            Xtr, Xval = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        else:
            Xtr, Xval = X_train[tr_idx], X_train[val_idx]
            
        ytr, yval = y_train[tr_idx], y_train[val_idx]
        
        m = lgb.LGBMClassifier(**params)
        m.fit(Xtr, ytr, eval_set=[(Xval, yval)],
              callbacks=[lgb.early_stopping(40, verbose=False),
                         lgb.log_evaluation(period=-1)])
                         
        aucs.append(roc_auc_score(yval, m.predict_proba(Xval)[:, 1]))
        
    return np.mean(aucs)

In [7]:
def objective_rf(trial):
    params = dict(
        n_estimators      = trial.suggest_int("n_estimators", 200, 1000),
        max_depth         = trial.suggest_int("max_depth", 3, 40),
        max_features      = trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3, 0.5, 0.7]),
        min_samples_split = trial.suggest_int("min_samples_split", 2, 30),
        min_samples_leaf  = trial.suggest_int("min_samples_leaf", 1, 20),
        max_samples       = trial.suggest_float("max_samples", 0.5, 1.0),
        class_weight      = "balanced",
        random_state      = SEED, 
        n_jobs            = -1,
    )
    
    aucs = []
    X_train = train_df[FEATURE_NAMES]
    X_test = test_df[FEATURE_NAMES]
    for tr_idx, val_idx in cv.split(X_train, y_train):
        if isinstance(X_train, pd.DataFrame):
            Xtr, Xval = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        else:
            Xtr = pd.DataFrame(X_train[tr_idx], columns=FEATURE_NAMES)
            Xval = pd.DataFrame(X_train[val_idx], columns=FEATURE_NAMES)
            
        ytr, yval = y_train[tr_idx], y_train[val_idx]
        
        m = Pipeline([
            ('preprocessor', clone(preprocessor)),
            ('rf', RandomForestClassifier(**params))
        ])
        
        m.fit(Xtr, ytr)
        aucs.append(roc_auc_score(yval, m.predict_proba(Xval)[:, 1]))
        
    return np.mean(aucs)

In [8]:
def build_xgb(p):
    return xgb.XGBClassifier(
        **p, eval_metric="auc", use_label_encoder=False,
        random_state=SEED, n_jobs=-1, verbosity=0)

def build_lgb(p):
    return lgb.LGBMClassifier(
        **p, is_unbalance=True, random_state=SEED, n_jobs=-1, verbosity=-1)

def build_rf(p):
    return RandomForestClassifier(
        **p, class_weight="balanced", random_state=SEED, n_jobs=-1)

LGB_FIT_KWARGS = dict(
    eval_set=None, 
    callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(period=-1)],
)

MODEL_CONFIGS = [
    {"name": "XGBoost",       "obj": objective_xgb, "build": build_xgb,
     "lgb": False},
    {"name": "LightGBM",      "obj": objective_lgb, "build": build_lgb,
     "lgb": True},
    {"name": "Random Forest",  "obj": objective_rf,  "build": build_rf,
     "lgb": False},
]

COLORS = {
    "XGBoost":        "#1565C0",
    "LightGBM":       "#2E7D32",
    "Random Forest":  "#6A1B9A",
}

results    = {}
estimators = []   

In [10]:
for cfg in MODEL_CONFIGS:
    name = cfg["name"]
    print(f"  [{name}]  Optuna {N_TRIALS} trials × {CV_FOLDS}-Fold StratifiedKFold")
    t0 = time.time()

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=15, n_min_trials=20),
    )
    study.optimize(cfg["obj"], n_trials=N_TRIALS, show_progress_bar=True)

    bp         = study.best_params
    best_cv_auc = study.best_value
    elapsed    = time.time() - t0

    print(f"\n  Thời gian    : {elapsed:.1f}s")
    print(f"  Best CV AUC  : {best_cv_auc:.4f}")
    print(f"  Best params  :\n{json.dumps(bp, indent=6)}")

    final_model = cfg["build"](bp)
    if cfg["lgb"]:
        final_model.fit(X_train, y_train,
                        eval_set=[(X_test, y_test)],
                        callbacks=[lgb.early_stopping(40, verbose=False),
                                   lgb.log_evaluation(period=-1)])
    else:
        final_model.fit(X_train, y_train)

    estimators.append((name.replace(" ", "_"), cfg["build"](bp)))

    oof_fit_kw = (dict(eval_set=[(X_train, y_train)],
                       callbacks=[lgb.early_stopping(40, verbose=False),
                                  lgb.log_evaluation(period=-1)])
                  if cfg["lgb"] else {})
    best_thr, oof_acc, oof_prob_train = find_best_threshold_oof(
        final_model, X_train, y_train, cfg["build"], bp, fit_kwargs=oof_fit_kw)

    test_prob = final_model.predict_proba(X_test)[:, 1]
    y_pred    = (test_prob >= best_thr).astype(int)

    test_acc  = accuracy_score(y_test, y_pred)
    test_auc  = roc_auc_score(y_test, test_prob)
    test_f1   = f1_score(y_test, y_pred)
    test_ap   = average_precision_score(y_test, test_prob)
    cm_val    = confusion_matrix(y_test, y_pred)
    report    = classification_report(y_test, y_pred, digits=4,
                                      target_names=["Not Converted", "Converted"])

    print(f"\n OOF Best Threshold : {best_thr:.3f}  (OOF Acc={oof_acc:.4f})")
    print(f" Test Accuracy      : {test_acc:.4f}")
    print(f" Test ROC-AUC       : {test_auc:.4f}")
    print(f" Test F1 (pos)      : {test_f1:.4f}")
    print(f" Avg Precision      : {test_ap:.4f}")
    print(f"\n{report}")

    results[name] = dict(
        model=final_model, study=study, best_params=bp,
        best_thr=best_thr, test_prob=test_prob, y_pred=y_pred,
        test_acc=test_acc, test_auc=test_auc, test_f1=test_f1,
        test_ap=test_ap, cm=cm_val, report=report,
    )

    joblib.dump(final_model,
                os.path.join(OUTPUT_DIR, f"model_{name.replace(' ','_')}.pkl"))
    with open(os.path.join(OUTPUT_DIR,
                           f"best_params_{name.replace(' ','_')}.json"), "w") as f:
        json.dump({"threshold": best_thr, **bp}, f, indent=4)
    print(f" Model + params đã lưu.")

  [XGBoost]  Optuna 200 trials × 5-Fold StratifiedKFold


Best trial: 193. Best value: 0.900392: 100%|██████████| 200/200 [01:26<00:00,  2.30it/s]



  Thời gian    : 86.9s
  Best CV AUC  : 0.9004
  Best params  :
{
      "n_estimators": 203,
      "max_depth": 3,
      "learning_rate": 0.13095700386160283,
      "subsample": 0.9718780959165109,
      "colsample_bytree": 0.9098586146196069,
      "colsample_bylevel": 0.8864477751213812,
      "min_child_weight": 1,
      "gamma": 0.9399664462694064,
      "reg_alpha": 2.3870291967285637e-08,
      "reg_lambda": 1.8013031951826498e-05,
      "scale_pos_weight": 1.6581074485089229
}

 OOF Best Threshold : 0.650  (OOF Acc=0.8259)
 Test Accuracy      : 0.8244
 Test ROC-AUC       : 0.8998
 Test F1 (pos)      : 0.7676
 Avg Precision      : 0.8635

               precision    recall  f1-score   support

Not Converted     0.8405    0.8782    0.8589       936
    Converted     0.7964    0.7409    0.7676       602

     accuracy                         0.8244      1538
    macro avg     0.8185    0.8095    0.8133      1538
 weighted avg     0.8232    0.8244    0.8232      1538

 Model + para

Best trial: 156. Best value: 0.900329: 100%|██████████| 200/200 [01:28<00:00,  2.27it/s]



  Thời gian    : 88.0s
  Best CV AUC  : 0.9003
  Best params  :
{
      "n_estimators": 586,
      "max_depth": 3,
      "num_leaves": 118,
      "learning_rate": 0.09163373390458887,
      "min_child_samples": 7,
      "subsample": 0.9576012277392844,
      "subsample_freq": 7,
      "colsample_bytree": 0.747451212124191,
      "reg_alpha": 0.001950117388921266,
      "reg_lambda": 4.5170094124846734e-05,
      "min_split_gain": 0.3319095817832958
}

 OOF Best Threshold : 0.590  (OOF Acc=0.8261)
 Test Accuracy      : 0.8290
 Test ROC-AUC       : 0.9028
 Test F1 (pos)      : 0.7807
 Avg Precision      : 0.8675

               precision    recall  f1-score   support

Not Converted     0.8576    0.8622    0.8599       936
    Converted     0.7839    0.7774    0.7807       602

     accuracy                         0.8290      1538
    macro avg     0.8208    0.8198    0.8203      1538
 weighted avg     0.8288    0.8290    0.8289      1538

 Model + params đã lưu.
  [Random Forest]  Optu

Best trial: 135. Best value: 0.898888: 100%|██████████| 200/200 [13:35<00:00,  4.08s/it]



  Thời gian    : 815.9s
  Best CV AUC  : 0.8989
  Best params  :
{
      "n_estimators": 499,
      "max_depth": 8,
      "max_features": "log2",
      "min_samples_split": 11,
      "min_samples_leaf": 1,
      "max_samples": 0.6734913812723163
}

 OOF Best Threshold : 0.590  (OOF Acc=0.8270)
 Test Accuracy      : 0.8296
 Test ROC-AUC       : 0.9019
 Test F1 (pos)      : 0.7791
 Avg Precision      : 0.8627

               precision    recall  f1-score   support

Not Converted     0.8532    0.8697    0.8614       936
    Converted     0.7911    0.7674    0.7791       602

     accuracy                         0.8296      1538
    macro avg     0.8222    0.8185    0.8202      1538
 weighted avg     0.8289    0.8296    0.8292      1538

 Model + params đã lưu.


Confusion Matrix

In [13]:
for name, res in results.items():
    cm = res["cm"]

    fig, ax = plt.subplots(figsize=(5.5, 4.8))

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["0", "1"],
                yticklabels=["0", "1"],
                linewidths=1.2, linecolor="white", ax=ax)

    ax.set_title(
        f"{name} - Confusion Matrix",
        fontsize=11, fontweight="bold"
    )

    ax.set_xlabel("Predicted Label", fontsize=11)
    ax.set_ylabel("True Label", fontsize=11)

    # metrics bên dưới
    ax.text(0.5, -0.25,
            f"Acc={res['test_acc']:.3f} | AUC={res['test_auc']:.3f} | "
            f"F1={res['test_f1']:.3f} | Thr={res['best_thr']:.2f}",
            transform=ax.transAxes,
            ha="center", fontsize=9)

    plt.tight_layout()

    path = os.path.join(OUTPUT_DIR, f"cm_{name.replace(' ','_')}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"CM ({name}) → {path}")

CM (XGBoost) → outputs\cm_XGBoost.png
CM (LightGBM) → outputs\cm_LightGBM.png
CM (Random Forest) → outputs\cm_Random_Forest.png


ROC + Precision-Recall

In [14]:
for name, res in results.items():
    color = COLORS.get(name, "gray")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    ax1.plot([0,1],[0,1],"k--", alpha=0.4, label="Random (AUC=0.50)")

    ax2.axhline(y=y_test.mean(), color="k", linestyle="--", alpha=0.4,
                label=f"Baseline (AP={y_test.mean():.3f})")

    fpr, tpr, _ = roc_curve(y_test, res["test_prob"])
    ax1.plot(fpr, tpr, lw=2.2, color=color,
             label=f"{name} (AUC={res['test_auc']:.4f})")

    cm = res["cm"]
    ax1.scatter(cm[0,1]/(cm[0].sum()+1e-9), cm[1,1]/(cm[1].sum()+1e-9),
                s=80, color=color, zorder=5, edgecolors="black", lw=0.8)

    prec, rec, _ = precision_recall_curve(y_test, res["test_prob"])
    ax2.plot(rec, prec, lw=2.2, color=color,
             label=f"{name} (AP={res['test_ap']:.4f})")

    for ax, title, xl, yl in [
        (ax1, f"ROC Curve - {name}", "False Positive Rate", "True Positive Rate"),
        (ax2, f"Precision-Recall Curve - {name}", "Recall", "Precision"),
    ]:
        ax.set_title(title, fontsize=13, fontweight="bold")
        ax.set_xlabel(xl, fontsize=12)
        ax.set_ylabel(yl, fontsize=12)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

    plt.tight_layout()

    path = os.path.join(OUTPUT_DIR, f"{name}_roc_pr.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"{name} → {path}")

XGBoost → outputs\XGBoost_roc_pr.png
LightGBM → outputs\LightGBM_roc_pr.png
Random Forest → outputs\Random Forest_roc_pr.png


TỔNG KẾT

In [15]:
print("  BẢNG TỔNG KẾT KẾT QUẢ")
rows = []
for name, res in results.items():
    cv_auc = f"{res['study'].best_value:.4f}" if res["study"] else "N/A (Ensemble)"
    rows.append({
        "Model"          : name,
        "Accuracy"       : f"{res['test_acc']:.4f}",
        "ROC-AUC"        : f"{res['test_auc']:.4f}",
        "F1 (positive)"  : f"{res['test_f1']:.4f}",
        "Avg Precision"  : f"{res['test_ap']:.4f}",
        "OOF Threshold"  : f"{res['best_thr']:.3f}",
        "CV AUC (Optuna)": cv_auc,
    })
df = pd.DataFrame(rows).set_index("Model")
print(df.to_string())

best_acc = max(results, key=lambda n: results[n]["test_acc"])
best_auc = max(results, key=lambda n: results[n]["test_auc"])
print(f"\nBest Accuracy : {best_acc}  →  {results[best_acc]['test_acc']:.4f}")
print(f"Best AUC      : {best_auc}  →  {results[best_auc]['test_auc']:.4f}")


  BẢNG TỔNG KẾT KẾT QUẢ
              Accuracy ROC-AUC F1 (positive) Avg Precision OOF Threshold CV AUC (Optuna)
Model                                                                                   
XGBoost         0.8244  0.8998        0.7676        0.8635         0.650          0.9004
LightGBM        0.8290  0.9028        0.7807        0.8675         0.590          0.9003
Random Forest   0.8296  0.9019        0.7791        0.8627         0.590          0.8989

Best Accuracy : Random Forest  →  0.8296
Best AUC      : LightGBM  →  0.9028
